![image.png](https://i.imgur.com/a3uAqnb.png)

# Out-of-Distribution Detection: Softmax Confidence versus Energy

- **Dataset**: [CIFAR-10](https://huggingface.co/datasets/uoft-cs/cifar10) as the known classes; test images of [CIFAR-100](https://huggingface.co/datasets/uoft-cs/cifar100), [SVHN](https://huggingface.co/datasets/ufldl-stanford/svhn) and [DTD](https://huggingface.co/datasets/tanganke/dtd) as unknown classes, from Hugging Face
- **Task**: out-of-distribution detection: flag images of classes outside the training label set
- **Model**: a ResNet-18 (about 11.2M parameters) trained from scratch; the detection scores need no training
- **Runtime**: Colab T4 GPU, about 6 minutes of training

In this lab, we will:

- see how confidently a trained classifier labels images of classes it has never seen
- compute three scores from its logits: the maximum softmax probability, the maximum logit and the energy
- implement AUROC and FPR95 and compare the scores on three unknown sets
- see which part of the logits each score uses, and set a rejection threshold

## What is out-of-distribution detection?

A softmax classifier trained on $K$ classes divides a total probability of 1 among the same $K$ labels for every input, so an image of anything else still gets one of them, often confidently. Images from the training distribution are **in-distribution (ID)**, or known; images of other classes are **out-of-distribution (OOD)**, or unknown. A detector adds a score $s(x)$, higher for familiar inputs, and a threshold $\lambda$: below it, the model answers "unknown".

The baseline score is the **maximum softmax probability** (MSP), $\max_k p(k \mid x)$. The **energy score** reads the logits before the softmax normalises them:

$$E(x) = -T \log \sum_{k=1}^{K} e^{z_k(x)/T}, \qquad s(x) = -E(x)$$

Where:
- $z(x) = W h(x) + b \in \mathbb{R}^K$ are the logits: the last linear layer (weights $W$, bias $b$) applied to the penultimate features $h(x)$; $p(k \mid x)$ is their softmax
- $T > 0$ is a temperature; we use $T = 1$, as Liu et al. (2020) recommend
- cross-entropy training lowers the energy of training images, so a high $-E(x)$ marks a familiar input

At $T = 1$, $\log \max_k p(k \mid x) = \max_k z_k(x) + E(x)$. Adding the same amount to every logit raises the first term and lowers the second equally, so the MSP ignores the overall level of the logits, which the energy keeps; this lab tests whether that level helps.

Paper: [Liu et al., 2020. Energy-based Out-of-distribution Detection](https://arxiv.org/abs/2010.03759)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub.utils import logging as hf_logging
from PIL import Image, ImageOps
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset

hf_logging.set_verbosity_error()   # the datasets are public: hide the notice that no Hugging Face token is set

In [ ]:
SEED = 42
N_TRAIN = 45_000        # CIFAR-10 training images to train on (the split has 50,000)
N_VAL = 5_000           # held-out CIFAR-10 training images: monitoring and the threshold
N_TEST = 10_000         # CIFAR-10 test images, the known (ID) test inputs
N_TEST_OOD = 10_000     # images taken from each unknown (OOD) test set, at most
IMG_SIZE = 32
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 512
EPOCHS = 20
LR = 0.1                # peak learning rate of the one-cycle schedule
WEIGHT_DECAY = 5e-4
TEMPERATURE = 1.0       # T in the energy score
TARGET_TPR = 0.95       # share of known images the threshold must accept
NUM_WORKERS = 2         # DataLoader worker processes; a Colab runtime has 2 CPU cores

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 on a T4, which has no bfloat16 units, with a gradient scaler against
# underflow; bfloat16 on an A100 or newer, which needs no scaler; off on the CPU.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
torch.backends.cudnn.benchmark = True   # every input is 32x32, so cuDNN can pick its fastest kernels once

## 1. Datasets

- **Known**: [CIFAR-10](https://huggingface.co/datasets/uoft-cs/cifar10) (Krizhevsky, 2009), 60,000 colour images of 32x32 pixels in 10 balanced classes, 50,000 for training and 10,000 for testing.
- **Unknown, near**: the 10,000 test images of [CIFAR-100](https://huggingface.co/datasets/uoft-cs/cifar100), 100 other classes from the same image collection, such as wolves and pickup trucks (CIFAR-10's "truck" covers only big trucks, says the [dataset page](https://www.cs.toronto.edu/~kriz/cifar.html)).
- **Unknown, far**: the 26,032 test images of [SVHN](https://huggingface.co/datasets/ufldl-stanford/svhn) (Netzer et al., 2011), house-number digits from Google Street View, and the 1,880 test images of [DTD](https://huggingface.co/datasets/tanganke/dtd) (Cimpoi et al., 2014), photos of textures in 47 categories ("Textures" below).
- **Licences**: not stated on the CIFAR cards; SVHN for non-commercial use only; DTD for research purposes.
- **What we use**: 45,000 CIFAR-10 images to train, 5,000 to validate, the 10,000 test images and up to 10,000 images of each unknown set.

In [ ]:
cifar10 = load_dataset("uoft-cs/cifar10")
CLASS_NAMES = cifar10["train"].features["label"].names
split = cifar10["train"].train_test_split(test_size=N_VAL, seed=SEED, stratify_by_column="label")
train_split = split["train"].shuffle(seed=SEED).select(range(min(N_TRAIN, len(split["train"]))))
val_split = split["test"]
test_split = cifar10["test"].shuffle(seed=SEED).select(range(min(N_TEST, len(cifar10["test"]))))
print(f"CIFAR-10: {len(train_split):,} train, {len(val_split):,} validation, {len(test_split):,} test | classes: {CLASS_NAMES}")

# name: (Hugging Face id, test file pattern, image column, label column)
OOD_SOURCES = {
    "CIFAR-100": ("uoft-cs/cifar100", "cifar100/test-*.parquet", "img", "fine_label"),
    "SVHN": ("ufldl-stanford/svhn", "cropped_digits/test-*.parquet", "image", "label"),
    "Textures": ("tanganke/dtd", "data/test-*.parquet", "image", "label"),
}
ood_splits = {}
for name, (repo, files, _, _) in OOD_SOURCES.items():
    # data_files fetches only the test file; no_checks stops the library from expecting the
    # other splits that the dataset card lists
    ds = load_dataset(repo, data_files={"test": files}, split="test", verification_mode="no_checks")
    ood_splits[name] = ds.shuffle(seed=SEED).select(range(min(N_TEST_OOD, len(ds))))
    print(f"{name}: {len(ds):,} test images, using {len(ood_splits[name]):,}")

We decode every image once into a `uint8` array of shape (N, 32, 32, 3), so that training reads pixels from memory. Texture photos are resized to 32 pixels on the shorter side and centre-cropped. Expected time: about a minute.

In [ ]:
def to_uint8_array(ds, image_column):
    """Decodes every image of a dataset split into an (N, IMG_SIZE, IMG_SIZE, 3) uint8 array."""
    images = np.empty((len(ds), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for i, row in enumerate(tqdm(ds, desc="decoding", leave=False)):
        img = row[image_column].convert("RGB")
        if img.size != (IMG_SIZE, IMG_SIZE):
            img = ImageOps.fit(img, (IMG_SIZE, IMG_SIZE), method=Image.Resampling.BILINEAR)
        images[i] = np.asarray(img)
    return images


x_train, y_train = to_uint8_array(train_split, "img"), np.array(train_split["label"])
x_val, y_val = to_uint8_array(val_split, "img"), np.array(val_split["label"])
x_test, y_test = to_uint8_array(test_split, "img"), np.array(test_split["label"])

ood_images, ood_labels, ood_label_names = {}, {}, {}
for name, (_, _, image_column, label_column) in OOD_SOURCES.items():
    ood_images[name] = to_uint8_array(ood_splits[name], image_column)
    ood_labels[name] = np.array(ood_splits[name][label_column])
    names = ood_splits[name].features[label_column].names
    ood_label_names[name] = ["crab" if n == "cra" else n for n in names]   # the CIFAR-100 card spells crab "cra"

print("x_train", x_train.shape, "| x_val", x_val.shape, "| x_test", x_test.shape)
for name, images in ood_images.items():
    print(f"{name}: {images.shape}")

### Known and unknown images

One row per set, each image with its own label. CIFAR-100 shares the image collection of CIFAR-10; the digits and textures do not.

In [ ]:
rows = [("CIFAR-10 (known)", x_test, y_test, CLASS_NAMES)]
rows += [(f"{name} (unknown)", ood_images[name], ood_labels[name], ood_label_names[name]) for name in ood_images]
n_cols = 8
fig, axes = plt.subplots(len(rows), n_cols, figsize=(1.6 * n_cols, 1.9 * len(rows)), squeeze=False)
for r, (title, images, labels, names) in enumerate(rows):
    for c in range(n_cols):
        axes[r, c].imshow(images[c])
        axes[r, c].set_title(names[labels[c]], fontsize=8)
        axes[r, c].axis("off")
    axes[r, 0].text(-0.15, 0.5, title, transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### Normalisation, augmentation and loaders

Every image, known or unknown, is normalised on the GPU with the channel statistics of 5,000 CIFAR-10 training images. Training images are also shifted by up to 4 pixels and flipped at random, by slicing a padded `uint8` copy, which keeps the CPU work small.

In [ ]:
sample = x_train[:5_000].astype(np.float32) / 255
MEAN = sample.mean(axis=(0, 1, 2)).tolist()
STD = sample.std(axis=(0, 1, 2)).tolist()
print(f"mean {np.round(MEAN, 4)} | std {np.round(STD, 4)}")

MEAN_T = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)


def normalize(images):
    """uint8 batch (B, 3, 32, 32) on DEVICE -> normalised float32 batch."""
    return (images.float() / 255 - MEAN_T) / STD_T


class ArrayDataset(Dataset):
    """Serves (image, label) pairs from a uint8 image array; the label is -1 for unlabelled images.
    With augment=True each image is shifted by up to 4 pixels and flipped left to right at random."""

    def __init__(self, images, labels=None, augment=False):
        self.images = torch.from_numpy(images).permute(0, 3, 1, 2).contiguous()   # (N, 3, 32, 32) uint8
        self.labels = torch.full((len(images),), -1) if labels is None else torch.as_tensor(labels)
        self.augment = augment
        if augment:
            self.padded = F.pad(self.images, (4, 4, 4, 4))                        # (N, 3, 40, 40), black border

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        if not self.augment:
            return self.images[i], self.labels[i]
        dy, dx = torch.randint(0, 9, (2,)).tolist()                               # corner of the 32x32 window
        image = self.padded[i, :, dy:dy + IMG_SIZE, dx:dx + IMG_SIZE]
        if torch.rand(()) < 0.5:
            image = image.flip(-1)
        return image, self.labels[i]


def make_eval_loader(images, labels=None):
    return DataLoader(ArrayDataset(images, labels), batch_size=EVAL_BATCH_SIZE,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")


# drop_last: an incomplete last batch could hold a single image, which BatchNorm cannot normalise in training mode
train_loader = DataLoader(ArrayDataset(x_train, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=NUM_WORKERS, pin_memory=DEVICE == "cuda")
val_loader = make_eval_loader(x_val, y_val)

images, labels = next(iter(train_loader))
print(images.shape, images.dtype, labels.shape)   # (BATCH_SIZE, 3, 32, 32) uint8, (BATCH_SIZE,)

## 2. Model

A randomly initialised ResNet-18 (He et al., 2016). Its ImageNet stem would shrink a 32x32 image to 8x8 before the first residual block, so we replace it with one 3x3 convolution of stride 1, as is usual for CIFAR-10. Average pooling gives the features $h(x)$, 512 numbers, and the last linear layer `fc` the logits $z(x) = W h(x) + b$.

In [ ]:
def make_resnet18(num_classes):
    """ResNet-18 for 32x32 inputs, randomly initialised."""
    model = torchvision.models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)   # (B, 64, 32, 32)
    model.maxpool = nn.Identity()
    # layer1 (B, 64, 32, 32) -> layer2 (B, 128, 16, 16) -> layer3 (B, 256, 8, 8) -> layer4 (B, 512, 4, 4)
    # -> avgpool and flatten: h(x), (B, 512) -> fc: z(x), (B, num_classes)
    return model


model = make_resnet18(len(CLASS_NAMES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
with torch.no_grad():
    print("logits:", model(normalize(images[:4].to(DEVICE))).shape)   # (4, 10)

## 3. Training

Plain cross-entropy, with nothing added for detection. SGD with Nesterov momentum and weight decay follows a one-cycle schedule, updated after every batch: the learning rate warms up to `LR` and then anneals towards zero, while the momentum moves the other way, between 0.95 and 0.85.

Expected time: about 6 minutes on a T4.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler):
    """One pass over the training set. Returns the mean loss and the accuracy."""
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    # a bar given the DataLoader itself would start a second, unused set of worker processes
    progress = tqdm(iter(loader), total=len(loader), desc="train", leave=False)
    for images, labels in progress:
        images, labels = normalize(images.to(DEVICE, non_blocking=True)), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(images)                  # (B, 10)
            loss = criterion(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()               # the scaler is a no-op when disabled
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
        progress.set_postfix(loss=f"{loss.item():.3f}")
    return total_loss / seen, correct / seen


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Mean loss and accuracy on a held-out loader."""
    model.eval()
    total_loss, correct, seen = 0.0, 0, 0
    for images, labels in loader:
        images, labels = normalize(images.to(DEVICE, non_blocking=True)), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
    return total_loss / seen, correct / seen

In [ ]:
criterion = nn.CrossEntropyLoss()
# Nesterov needs a momentum here; OneCycleLR then replaces it, cycling it between 0.95 and 0.85
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))
print(f"{len(train_loader):,} optimizer steps per epoch, {EPOCHS} epochs")

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
start = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    for key, value in zip(history, (train_loss, train_acc, val_loss, val_acc)):
        history[key].append(value)
    print(f"Epoch {epoch}/{EPOCHS}: train loss {train_loss:.4f}, acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f}, acc {val_acc:.3f} | {time.time() - start:.0f}s elapsed")

## 4. Results

The training curves first.

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(epochs, history["train_loss"], marker="o", label="train")
ax_loss.plot(epochs, history["val_loss"], marker="o", label="validation")
ax_loss.set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
ax_loss.legend()
ax_acc.plot(epochs, history["train_acc"], marker="o", label="train")
ax_acc.plot(epochs, history["val_acc"], marker="o", label="validation")
ax_acc.set(xlabel="epoch", ylabel="accuracy", title="Accuracy")
ax_acc.legend()
plt.tight_layout()
plt.show()

Every score below is a function of the logits, so we compute them once per set and print the test accuracy.

In [ ]:
@torch.no_grad()
def predict_logits(model, images):
    """Logits (N, K) of a uint8 image array, in float32 on the CPU."""
    model.eval()
    out = []
    for batch, _ in make_eval_loader(images):
        # float32, not float16: scores that differ in the third decimal must not round to the same value
        out.append(model(normalize(batch.to(DEVICE, non_blocking=True))).float().cpu())
    return torch.cat(out)


val_logits = predict_logits(model, x_val)
test_logits = predict_logits(model, x_test)
ood_logits = {name: predict_logits(model, images) for name, images in ood_images.items()}

test_acc = (test_logits.argmax(dim=1).numpy() == y_test).mean()
print(f"CIFAR-10 test accuracy: {test_acc:.3f} on {len(y_test):,} images")

## 5. Confident Answers on Unknown Images

The classifier cannot answer "none of these". We count the images with an MSP above 0.9 and 0.99 in each set, and show the unknown images it is surest about, with their true label, prediction and MSP.

In [ ]:
sets = {"CIFAR-10 (known)": test_logits} | {f"{name} (unknown)": logits for name, logits in ood_logits.items()}
for title, logits in sets.items():
    msp = F.softmax(logits.double(), dim=1).max(dim=1).values
    print(f"{title:<20} mean MSP {msp.mean():.3f} | MSP > 0.9: {(msp > 0.9).double().mean():6.1%} | "
          f"MSP > 0.99: {(msp > 0.99).double().mean():6.1%}")

n_show = 6
fig, axes = plt.subplots(len(ood_logits), n_show, figsize=(1.9 * n_show, 2.3 * len(ood_logits)), squeeze=False)
for r, (name, logits) in enumerate(ood_logits.items()):
    msp, pred = F.softmax(logits.double(), dim=1).max(dim=1)
    for c, i in enumerate(torch.argsort(msp, descending=True)[:n_show].tolist()):
        axes[r, c].imshow(ood_images[name][i])
        axes[r, c].set_title(f"{ood_label_names[name][ood_labels[name][i]]}\nas {CLASS_NAMES[pred[i]]}, MSP {msp[i]:.3f}",
                             fontsize=8)
        axes[r, c].axis("off")
    axes[r, 0].text(-0.15, 0.5, name, transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=9)
plt.tight_layout()
plt.show()

**What to notice**

Many unknown images get an MSP above 0.9, and some above 0.99. The surest mistakes on CIFAR-100 are often near misses, such as vehicles taken for trucks, but digits and textures can also score close to 1. Known images reach these levels far more often: the MSP carries a signal, only a noisy one.

## 6. Three Scores from the Same Logits

Each score is higher for familiar images:

- **MSP**: $\max_k p(k \mid x)$ (Hendrycks and Gimpel, 2017)
- **MaxLogit**: $\max_k z_k(x)$ (Hendrycks et al., 2022)
- **Energy**: $-E(x) = T \log \sum_k e^{z_k(x)/T}$; at $T = 1$, $\max_k z_k \leq -E(x) \leq \max_k z_k + \log K$, so $-E(x)$ is the MaxLogit plus at most $\log K$

The last line checks $\log \max_k p(k \mid x) = \max_k z_k(x) + E(x)$ on the test images.

In [ ]:
# The scores work in float64, so that MSP values very close to 1 stay distinct.
def msp_score(logits):
    """Maximum softmax probability."""
    return F.softmax(logits.double(), dim=1).max(dim=1).values


def max_logit_score(logits):
    """Largest logit."""
    return logits.double().max(dim=1).values


def energy_score(logits, T=TEMPERATURE):
    """Negative energy -E(x) = T log sum_k exp(z_k / T); higher means more familiar."""
    return T * torch.logsumexp(logits.double() / T, dim=1)


SCORES = {"MSP": msp_score, "MaxLogit": max_logit_score, "Energy": energy_score}

residual = torch.log(msp_score(test_logits)) - (max_logit_score(test_logits) - energy_score(test_logits, T=1.0))
print(f"largest |log MSP - (max logit + E)| over {len(residual):,} test images: {residual.abs().max():.2e}")

## 7. Measuring a Detector: AUROC and FPR95

Known images are the positive class: a threshold $\lambda$ accepts the images with $s(x) \geq \lambda$, the **true positive rate** (TPR) is the share of known images accepted and the **false positive rate** (FPR) that of unknown images.

- **AUROC**, the area under the ROC curve (TPR against FPR as $\lambda$ sweeps), is the probability that a random known image outscores a random unknown one: 1 is perfect, 0.5 is chance.
- **FPR95** is the FPR at the highest threshold that accepts 95% of the known images; lower is better.

We check both against scikit-learn, then score every set (in percent).

In [ ]:
def auroc(id_scores, ood_scores):
    """P(random ID score > random OOD score), ties counted one half, from the rank-sum statistic."""
    ranks = pd.Series(np.concatenate([id_scores, ood_scores])).rank().to_numpy()   # 1 = lowest; ties share their mean rank
    n_id, n_ood = len(id_scores), len(ood_scores)
    # the ID rank sum minus its smallest possible value counts the (ID, OOD) pairs in which the ID score is higher
    return (ranks[:n_id].sum() - n_id * (n_id + 1) / 2) / (n_id * n_ood)


def threshold_at_tpr(id_scores, tpr=TARGET_TPR):
    """The highest threshold that accepts at least a share `tpr` of the ID scores: the k-th highest of them."""
    k = int(np.ceil(tpr * len(id_scores)))
    return np.sort(id_scores)[::-1][k - 1]


def fpr_at_tpr(id_scores, ood_scores, tpr=TARGET_TPR):
    """Share of OOD scores at or above that threshold."""
    return np.mean(ood_scores >= threshold_at_tpr(id_scores, tpr))


name = next(iter(ood_logits))
s_id, s_ood = energy_score(test_logits).numpy(), energy_score(ood_logits[name]).numpy()
y_true = np.r_[np.ones(len(s_id)), np.zeros(len(s_ood))]
fpr, tpr, _ = roc_curve(y_true, np.r_[s_id, s_ood])
print(f"{name}, energy | AUROC: ours {auroc(s_id, s_ood):.4f}, scikit-learn {roc_auc_score(y_true, np.r_[s_id, s_ood]):.4f}")
# scikit-learn has no FPR95: we read its ROC curve at the first point with a TPR of at least 95%
print(f"{name}, energy | FPR95: ours {fpr_at_tpr(s_id, s_ood):.4f}, scikit-learn {fpr[np.searchsorted(tpr, TARGET_TPR)]:.4f}")

results = []
for ood_name, logits in ood_logits.items():
    for score_name, score_fn in SCORES.items():
        s_id, s_ood = score_fn(test_logits).numpy(), score_fn(logits).numpy()
        results.append({"unknown set": ood_name, "score": score_name,
                        "AUROC": 100 * auroc(s_id, s_ood), "FPR95": 100 * fpr_at_tpr(s_id, s_ood)})
table = pd.DataFrame(results).pivot(index="score", columns="unknown set", values=["AUROC", "FPR95"])
table = table.loc[list(SCORES)].reindex(columns=list(ood_logits), level=1)
table.round(1)

Where MSP and energy put known and unknown images (the MSP row on a log scale). The dashed line accepts 95% of the known test images; the share of unknown images to its right is the FPR95.

In [ ]:
fig, axes = plt.subplots(2, len(ood_logits), figsize=(5 * len(ood_logits), 7), squeeze=False)
for c, (ood_name, logits) in enumerate(ood_logits.items()):
    for r, score_name in enumerate(["MSP", "Energy"]):
        s_id, s_ood = SCORES[score_name](test_logits).numpy(), SCORES[score_name](logits).numpy()
        bins = np.linspace(min(s_id.min(), s_ood.min()), max(s_id.max(), s_ood.max()), 50)
        log = score_name == "MSP"          # most known images share the last MSP bin; a log scale shows the rest
        ax = axes[r, c]
        ax.hist(s_id, bins=bins, alpha=0.6, density=True, log=log, label="CIFAR-10 (known)")
        ax.hist(s_ood, bins=bins, alpha=0.6, density=True, log=log, label=f"{ood_name} (unknown)")
        ax.axvline(threshold_at_tpr(s_id), color="black", linestyle="--", linewidth=1, label="accepts 95% of known")
        ax.set(xlabel="MSP" if log else "negative energy, -E(x)", ylabel="density (log scale)" if log else "density",
               title=f"{score_name}: AUROC {100 * auroc(s_id, s_ood):.1f}, FPR95 {100 * fpr_at_tpr(s_id, s_ood):.1f}")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**What to notice**

The MaxLogit and energy rows are close: by the identity of section 6, $-E(x) = \max_k z_k(x) - \log \text{MSP}$, and the log-MSP is near 0 for most images. On CIFAR-100 both beat the MSP, by a wide margin on FPR95; on the textures they usually keep a smaller lead. On SVHN the winner changes between training runs (Exercise 2), and AUROC and FPR95 can disagree. The far sets are not much easier than CIFAR-100 here. The paper also found the MSP ahead on AUROC for SVHN and textures (Liu et al., 2020, Table 1).

## 8. What Each Score Keeps

Adding a constant to every logit leaves the MSP unchanged but raises the MaxLogit and $-E(x)$ by that constant: the MSP reads only differences between logits, mostly the top-two gap, and the energy mostly the level of the largest. We check this on one test image, count the unknown images that only one score rejects, and plot up to 2,000 images per set by these two quantities.

In [ ]:
i = int(msp_score(test_logits).argmin())        # the least confident test image, so its MSP is far from 1
for shift in [0.0, 5.0]:
    z = test_logits[i:i + 1] + shift
    print(f"logits + {shift:.0f}: MSP {msp_score(z).item():.4f} | MaxLogit {max_logit_score(z).item():.3f} | "
          f"-E(x) {energy_score(z, T=1.0).item():.3f}")


def top2_gap(logits):
    """Largest logit minus the second largest."""
    top2 = logits.double().topk(2, dim=1).values
    return top2[:, 0] - top2[:, 1]


# unknown images rejected by only one of the two scores, each at the threshold that accepts 95% of known images
for ood_name, logits in ood_logits.items():
    by_energy = energy_score(logits).numpy() < threshold_at_tpr(energy_score(test_logits).numpy())
    by_msp = msp_score(logits).numpy() < threshold_at_tpr(msp_score(test_logits).numpy())
    print(f"{ood_name:<10} rejected by the energy only: {np.mean(by_energy & ~by_msp):6.1%} | "
          f"by the MSP only: {np.mean(by_msp & ~by_energy):6.1%}")

x_line = threshold_at_tpr(max_logit_score(test_logits).numpy())   # 95% of known images have a larger largest logit
y_line = threshold_at_tpr(top2_gap(test_logits).numpy())          # and 95% a larger gap
rng = np.random.default_rng(SEED)
id_idx = rng.choice(len(test_logits), size=min(2_000, len(test_logits)), replace=False)
fig, axes = plt.subplots(1, len(ood_logits), figsize=(5 * len(ood_logits), 4.5), sharex=True, sharey=True, squeeze=False)
for ax, (ood_name, logits) in zip(axes[0], ood_logits.items()):
    ood_idx = rng.choice(len(logits), size=min(2_000, len(logits)), replace=False)
    ax.scatter(max_logit_score(logits[ood_idx]), top2_gap(logits[ood_idx]), s=4, alpha=0.4, color="tab:orange",
               label=f"{ood_name} (unknown)")
    ax.scatter(max_logit_score(test_logits[id_idx]), top2_gap(test_logits[id_idx]), s=4, alpha=0.25, color="tab:blue",
               label="CIFAR-10 (known)")
    ax.axvline(x_line, color="black", linestyle="--", linewidth=1, label="95% of known lie to the right")
    ax.axhline(y_line, color="black", linestyle=":", linewidth=1, label="95% of known lie above")
    ax.set(xlabel="largest logit (MaxLogit; -E(x) is close)", ylabel="top-two logit gap (what the MSP mostly reads)",
           title=f"CIFAR-10 vs {ood_name}")
    ax.legend(fontsize=8, markerscale=3)
plt.tight_layout()
plt.show()

**What to notice**

The shift leaves the MSP unchanged and raises the other two scores by exactly 5. An energy threshold is close to a vertical line here and an MSP threshold to a horizontal one: unknown images left of the dashed line but above the dotted one are roughly those only the energy rejects, and those right of it but below, those only the MSP rejects. On CIFAR-100 the first group is larger, which is why the energy wins there.

## 9. Choosing a Threshold

FPR95 sets its threshold on the test images themselves; a deployed detector fixes it beforehand, so we take the energy score that 95% of the validation images reach. Then we check what it accepts, compare the accuracy on accepted and rejected known images, and show unknown images that get through.

In [ ]:
threshold = float(threshold_at_tpr(energy_score(val_logits).numpy()))
print(f"threshold on -E(x): {threshold:.3f} (accepts {TARGET_TPR:.0%} of the validation images)")

test_scores = energy_score(test_logits).numpy()
accepted = test_scores >= threshold
correct = test_logits.argmax(dim=1).numpy() == y_test
print(f"{'CIFAR-10 (known)':<22} {accepted.mean():6.1%} accepted")
for ood_name, logits in ood_logits.items():
    print(f"{ood_name + ' (unknown)':<22} {(energy_score(logits).numpy() >= threshold).mean():6.1%} accepted")
print(f"accuracy on accepted known images: {correct[accepted].mean():.3f} ({accepted.sum():,} images)")
if (~accepted).any():
    print(f"accuracy on rejected known images: {correct[~accepted].mean():.3f} ({(~accepted).sum():,} images)")

n_show = 6
fig, axes = plt.subplots(len(ood_logits), n_show, figsize=(1.9 * n_show, 2.3 * len(ood_logits)), squeeze=False)
for r, (ood_name, logits) in enumerate(ood_logits.items()):
    passed = np.flatnonzero(energy_score(logits).numpy() >= threshold)[:n_show]
    pred = logits.argmax(dim=1)
    for c in range(n_show):
        axes[r, c].axis("off")
        if c < len(passed):
            i = passed[c]
            axes[r, c].imshow(ood_images[ood_name][i])
            axes[r, c].set_title(f"{ood_label_names[ood_name][ood_labels[ood_name][i]]}\nas {CLASS_NAMES[pred[i]]}", fontsize=8)
    axes[r, 0].text(-0.15, 0.5, f"{ood_name}\naccepted", transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=9)
plt.tight_layout()
plt.show()

**What to notice**

The threshold accepts close to 95% of the known test images, and the known images it rejects are classified far less accurately than the accepted ones. Yet more than half of every unknown set gets through, each with a wrong label: at 95% TPR this detector stays weak.

## 10. Save and Reload

The detector is the classifier plus one number: we save both, reload them and check the scores.

In [ ]:
torch.save({"model": model.state_dict(), "threshold": threshold}, "resnet18_cifar10_ood.pth")

checkpoint = torch.load("resnet18_cifar10_ood.pth", map_location=DEVICE)
reloaded = make_resnet18(len(CLASS_NAMES)).to(DEVICE)
reloaded.load_state_dict(checkpoint["model"])
reloaded_scores = energy_score(predict_logits(reloaded, x_test)).numpy()
print(f"largest score difference after reloading: {np.abs(reloaded_scores - test_scores).max():.2e}")
print(f"known test images accepted after reloading: {(reloaded_scores >= checkpoint['threshold']).mean():.1%}")

## Summary

- A softmax classifier labels every image, often confidently, even images of classes it never saw.
- A detector is a score plus a threshold set on known images, measured by AUROC and FPR95.
- The MSP keeps only differences between logits; the energy, close to the MaxLogit, also keeps their level, which helped on CIFAR-100.
- Score rankings change with the unknown set and the training run, so comparisons need several of each.

## Exercises

1. **Temperature.** Put `TEMPERATURE = 0.1` first in the section 6 code cell, run from there, then try 10 and 100. What does $-E(x)$ become as $T \to 0$, and as $T$ grows?
2. **Another training run.** Keep the table (`table_42 = table.copy()`), set `SEED = 0` and run everything again (about 9 minutes on a T4). Which entries keep their ranking?
3. **The model's own mistakes.** With `correct` from section 9, compute `auroc(s[correct], s[~correct])` for each score `s` of the test images. Is the best score the same as for unknown images?
4. **Another unknown set.** Add `"MNIST": ("ylecun/mnist", "mnist/test-*.parquet", "image", "label")` to `OOD_SOURCES`, re-run section 1, skip sections 2 and 3 to keep the trained model, and re-run from the logits cell of section 4. Are handwritten digits easier to reject than house numbers? Guess first.

## Further Reading

- [Hendrycks and Gimpel, 2017. A Baseline for Detecting Misclassified and Out-of-Distribution Examples in Neural Networks](https://arxiv.org/abs/1610.02136)
- [Liu et al., 2020. Energy-based Out-of-distribution Detection](https://arxiv.org/abs/2010.03759)
- [Hendrycks et al., 2022. Scaling Out-of-Distribution Detection for Real-World Settings](https://arxiv.org/abs/1911.11132)

### Contributed by: Ali Habibullah